In [ ]:
# scripts/make_dashboard.py
"""
Dashboard builder (static HTML, no server)
==========================================
Inputs (repo-relative):
- data/processed/data_clean.csv

Outputs:
- docs/dashboard.html                   (interactive Plotly dashboard)
- outputs/dashboard_country_agg.csv     (country-level aggregates)
- outputs/dashboard_wg_totals.csv       (WG totals, if WG columns exist)
- docs/downloads/country_agg.csv        (same as above, for GitHub Pages)
- docs/downloads/wg_totals.csv          (same as above, if WG columns exist)

Notes
-----
- Uses Plotly to embed multiple charts into a single static HTML page that
  works on GitHub Pages (no Dash server).
- Re-uses the same column detection and boolean coercion logic as your map script.
"""

from __future__ import annotations
from pathlib import Path
import sys, re, shutil
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio


# --------------------------- repo root detection --------------------------- #
def _find_repo_root() -> Path:
    """Work in scripts, terminals, and notebooks."""
    try:
        here = Path(__file__).resolve()
        return here.parent.parent  # …/scripts → repo root
    except NameError:
        cwd = Path.cwd().resolve()
        if (cwd / "data").is_dir() and (cwd / "scripts").is_dir():
            return cwd
        if cwd.name == "scripts" and (cwd.parent / "data").is_dir():
            return cwd.parent
        cur = cwd
        for _ in range(5):
            if (cur / ".git").is_dir() or ((cur / "data").is_dir() and (cur / "scripts").is_dir()):
                return cur
            cur = cur.parent
        return cwd


ROOT      = _find_repo_root()
CSV_PATH  = ROOT / "data" / "processed" / "data_clean.csv"
DOCS_DIR  = ROOT / "docs"
OUT_DIR   = ROOT / "outputs"
DL_DIR    = DOCS_DIR / "downloads"  # files inside docs/ so GH Pages can serve them
DOCS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
DL_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root →", ROOT)

if not CSV_PATH.exists():
    sys.exit(f"ERROR: {CSV_PATH.relative_to(ROOT)} not found. Run scripts/make_eda.py first.")


# ------------------------------ helpers ----------------------------------- #
def yn_to_bool(s: pd.Series) -> pd.Series:
    """Coerce common yes/no-ish tokens to booleans."""
    return (
        s.astype(str)
         .str.strip().str.lower()
         .map({"y": True, "yes": True, "member": True, "1": True, "true": True, "t": True, "x": True,
               "n": False, "no": False, "0": False, "false": False, "" : False})
         .fillna(False)
    )


# ------------------------------ load/clean -------------------------------- #
df = pd.read_csv(CSV_PATH, low_memory=False)

# normalize NBSP + trim object strings (safe)
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].astype(str).str.replace("\u00A0", " ", regex=False).str.strip()

# column detection
country_col = "country_clean" if "country_clean" in df.columns else ("country" if "country" in df.columns else None)
if not country_col:
    sys.exit("ERROR: no 'country_clean' or 'country' column in data_clean.csv")

mc_col   = "mc_member"  if "mc_member"  in df.columns else None
core_col = "core_group" if "core_group" in df.columns else None
wg_cols  = [c for c in df.columns if re.match(r"(?i)^wg\d", c)]
wg_cols  = sorted(wg_cols, key=lambda x: int(re.findall(r"\d+", x)[0]) if re.findall(r"\d+", x) else 99)
wg_member_col = "wg_member" if "wg_member" in df.columns else None

# coerce flags
if mc_col:
    if df[mc_col].dtype != bool:
        df[mc_col] = yn_to_bool(df[mc_col])
else:
    df["mc_member"] = False
    mc_col = "mc_member"

if core_col:
    if df[core_col].dtype != bool:
        df[core_col] = yn_to_bool(df[core_col])
else:
    df["core_group"] = False
    core_col = "core_group"

for c in wg_cols:
    if df[c].dtype != bool:
        df[c] = yn_to_bool(df[c])

df["any_wg"] = df[wg_cols].any(axis=1) if wg_cols else False
if wg_member_col:
    df["any_wg"] = df["any_wg"] | yn_to_bool(df[wg_member_col])


# ------------------------------ aggregates -------------------------------- #
# Per-country aggregates
agg_parts = {
    "Country":      (country_col, "first"),
    "Total":        (country_col, "size"),
    "MC":           (mc_col, "sum"),
    "Core":         (core_col, "sum"),
    "Any WG":       ("any_wg", "sum"),
}
for c in wg_cols:
    agg_parts[c] = (c, "sum")

country_agg = df.groupby(country_col, as_index=False).agg(**agg_parts)

# clean types
for c in ["Total", "MC", "Core", "Any WG"] + wg_cols:
    if c in country_agg.columns:
        country_agg[c] = country_agg[c].fillna(0).astype(int)

# Totals for KPI
kpi_total_people = int(df.shape[0])
kpi_countries    = int(country_agg["Country"].nunique())
kpi_mc_total     = int(country_agg["MC"].sum())
kpi_core_total   = int(country_agg["Core"].sum())
kpi_any_wg_total = int(country_agg["Any WG"].sum())

# WG Totals
if wg_cols:
    wg_totals = country_agg[wg_cols].sum().rename_axis("wg").reset_index(name="count")
else:
    wg_totals = pd.DataFrame(columns=["wg", "count"])

# Export aggregates for reuse (both to outputs/ and docs/downloads/)
out_country_csv = OUT_DIR / "dashboard_country_agg.csv"
country_agg.to_csv(out_country_csv, index=False, encoding="utf-8-sig")
shutil.copyfile(out_country_csv, DL_DIR / "country_agg.csv")
print("Saved →", out_country_csv.relative_to(ROOT))
print("Saved →", (DL_DIR / "country_agg.csv").relative_to(ROOT))

if not wg_totals.empty:
    out_wg_csv = OUT_DIR / "dashboard_wg_totals.csv"
    wg_totals.to_csv(out_wg_csv, index=False, encoding="utf-8-sig")
    shutil.copyfile(out_wg_csv, DL_DIR / "wg_totals.csv")
    print("Saved →", out_wg_csv.relative_to(ROOT))
    print("Saved →", (DL_DIR / "wg_totals.csv").relative_to(ROOT))


# ------------------------------ figures ----------------------------------- #
# 1) Top countries by Total
topN = 15
top_countries = (country_agg.sort_values("Total", ascending=False).head(topN))
fig_top_total = px.bar(
    top_countries,
    x="Total", y="Country",
    orientation="h",
    title=f"Top {min(topN, len(country_agg))} Countries by Total People",
)
fig_top_total.update_layout(yaxis={"categoryorder": "total ascending"}, margin=dict(l=10, r=10, t=60, b=10))

# 2) MC by country (top 15)
fig_top_mc = px.bar(
    country_agg.sort_values("MC", ascending=False).head(topN),
    x="MC", y="Country",
    orientation="h",
    title=f"Top {min(topN, len(country_agg))} Countries by MC Members",
)
fig_top_mc.update_layout(yaxis={"categoryorder": "total ascending"}, margin=dict(l=10, r=10, t=60, b=10))

# 3) Core by country (top 15)
fig_top_core = px.bar(
    country_agg.sort_values("Core", ascending=False).head(topN),
    x="Core", y="Country",
    orientation="h",
    title=f"Top {min(topN, len(country_agg))} Countries by Core Group Members",
)
fig_top_core.update_layout(yaxis={"categoryorder": "total ascending"}, margin=dict(l=10, r=10, t=60, b=10))

# 4) WG breakdown for top 10 countries by Total (grouped bars)
if wg_cols:
    top10 = country_agg.sort_values("Total", ascending=False).head(10).copy()
    melt_cols = ["Country"] + wg_cols
    long_wg = top10[melt_cols].melt(id_vars="Country", var_name="WG", value_name="Count")
    # nicer WG labels for chart (“WG 1”, “WG 2”, …)
    def _pretty(w):
        m = re.search(r"\d+", w)
        return f"WG {m.group(0)}" if m else w
    long_wg["WG"] = long_wg["WG"].map(_pretty)

    fig_wg_grouped = px.bar(
        long_wg, x="Count", y="Country", color="WG",
        orientation="h",
        barmode="group",
        title="Working Group membership — Top 10 countries (grouped)",
    )
    fig_wg_grouped.update_layout(yaxis={"categoryorder": "total ascending"},
                                 legend_title="Working Group",
                                 margin=dict(l=10, r=10, t=60, b=10))
else:
    fig_wg_grouped = None

# 5) Any WG share (pie)
has_wg = int((df["any_wg"] == True).sum())   # noqa: E712
no_wg  = int((df["any_wg"] == False).sum())  # noqa: E712
fig_anywg_pie = px.pie(
    pd.DataFrame({"label": ["Any WG", "No WG"], "count": [has_wg, no_wg]}),
    names="label", values="count", title="Share of people with any WG membership",
    hole=0.35
)
fig_anywg_pie.update_traces(textposition="inside", textinfo="percent+label")

# 6) WG correlation heatmap (if WG columns exist)
if wg_cols and len(wg_cols) >= 2:
    # compute correlation on boolean (converted to int)
    corr = df[wg_cols].astype(int).corr()
    # pretty labels
    pretty_cols = [f"WG {re.search(r'\\d+', c).group(0)}" if re.search(r"\\d+", c) else c for c in corr.columns]
    fig_wg_corr = go.Figure(data=go.Heatmap(
        z=corr.values, x=pretty_cols, y=pretty_cols, zmin=-1, zmax=1, colorscale="RdBu"
    ))
    fig_wg_corr.update_layout(title="Correlation of WG memberships (row-level)", margin=dict(l=10, r=10, t=60, b=10))
else:
    fig_wg_corr = None


# ------------------------------ assemble HTML ------------------------------ #
# We'll embed Plotly JS from CDN once; then each fig as a DIV (full_html=False).
PLOTLY_CDN = '<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>'

def _div(fig: go.Figure, include_js: bool = False) -> str:
    return pio.to_html(fig, include_plotlyjs=include_js, full_html=False, default_width="100%", default_height="100%")

kpi_html = f"""
<section class="kpis">
  <div class="kpi"><div class="kpi-num">{kpi_total_people:,}</div><div class="kpi-label">Total people</div></div>
  <div class="kpi"><div class="kpi-num">{kpi_countries:,}</div><div class="kpi-label">Countries</div></div>
  <div class="kpi"><div class="kpi-num">{kpi_mc_total:,}</div><div class="kpi-label">MC members</div></div>
  <div class="kpi"><div class="kpi-num">{kpi_core_total:,}</div><div class="kpi-label">Core group</div></div>
  <div class="kpi"><div class="kpi-num">{kpi_any_wg_total:,}</div><div class="kpi-label">Any WG</div></div>
</section>
"""

grid_html = f"""
<section class="grid">
  <div class="card">{_div(fig_top_total, include_js=True)}</div>
  <div class="card">{_div(fig_anywg_pie)}</div>
  <div class="card">{_div(fig_top_mc)}</div>
  <div class="card">{_div(fig_top_core)}</div>
  {"<div class='card'>" + _div(fig_wg_grouped) + "</div>" if fig_wg_grouped else ""}
  {"<div class='card'>" + _div(fig_wg_corr)    + "</div>" if fig_wg_corr    else ""}
</section>
"""

table_preview = (
    country_agg.sort_values("Total", ascending=False)
               .head(25)
               .to_html(index=False, classes="datatable", border=0)
)

page_html = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <title>Agrifood Evidence — Dashboard</title>
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  {PLOTLY_CDN}
  <style>
    :root {{ --accent:#1F5FCC; --text:#222; --muted:#666; --bg:#fff; --card:#f6f8fb; }}
    * {{ box-sizing:border-box; }}
    body {{ margin:0; font-family: system-ui,-apple-system,Segoe UI,Roboto,Arial,sans-serif; color:var(--text); background:var(--bg);}}
    .container {{ max-width: 1200px; margin: 1.5rem auto; padding: 0 1rem; }}
    header h1 {{ margin:0; }}
    header .meta {{ color:var(--muted); margin:.25rem 0 1rem; }}
    .kpis {{ display:grid; grid-template-columns: repeat(auto-fit,minmax(160px,1fr)); gap:12px; margin: 1rem 0 1.25rem; }}
    .kpi {{ background:var(--card); border:1px solid #e5e7eb; border-radius:12px; padding:14px; }}
    .kpi-num {{ font-size:1.8rem; font-weight:700; line-height:1; }}
    .kpi-label {{ color:var(--muted); font-size:.95rem; }}
    .grid {{ display:grid; gap:14px; grid-template-columns: repeat(auto-fit,minmax(320px,1fr)); }}
    .card {{ background:#fff; border:1px solid #e5e7eb; border-radius:12px; padding:8px; min-height: 340px; }}
    h2 {{ margin:1.5rem 0 .5rem; }}
    .datatable {{ border-collapse:collapse; width:100%; }}
    .datatable th, .datatable td {{ border-bottom:1px solid #eee; padding:6px 8px; text-align:left; }}
    .downloads a {{ color:var(--accent); text-decoration:none; }}
  </style>
</head>
<body>
  <main class="container">
    <header>
      <h1>Agrifood Evidence — Dashboard</h1>
      <p class="meta">Interactive charts from <code>data/processed/data_clean.csv</code></p>
    </header>

    {kpi_html}
    {grid_html}

    <section>
      <h2>Country table (top 25 by Total)</h2>
      <div class="downloads">
        Download data:
        <a href="downloads/country_agg.csv">country_agg.csv</a>
        {" • <a href='downloads/wg_totals.csv'>wg_totals.csv</a>" if not wg_totals.empty else ""}
      </div>
      {table_preview}
    </section>
  </main>
</body>
</html>
"""

out_path = DOCS_DIR / "dashboard.html"
out_path.write_text(page_html, encoding="utf-8")
print("Saved →", out_path.relative_to(ROOT))


Repo root → C:\Users\James\Documents\GitHub\evidence-map-agrifood
Saved → outputs\dashboard_country_agg.csv
Saved → outputs\dashboard_wg_totals.csv
Saved → docs\dashboard.html
